In [1]:
# import the necessary libraries to execute this code
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from collections import defaultdict
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
from scipy.cluster import hierarchy
import pickle

# Impact of input feature on model performance - LightGBM

In [2]:
datafile = "C:/chem papers/Dataset_17_feat.xlsx"
df = pd.read_excel(datafile)

with open('Trained_models/17_feat_LGBM_model.pkl', 'rb') as file:  
    model = pickle.load(file)

In [4]:
with open('C:/chem papers/TRAINED_MODELS/17_feat_LGBM_model.pkl', 'rb') as file:  
    df = pickle.load(file)
    

In [9]:
import pandas as pd

df = pd.read_csv("C:/chem papers/Dataset_17_feat.csv")   # or however you originally loaded it

X = df.drop(['Experimental_index','DP_Group','Release'], axis='columns')
X_features = X.copy()
stdScale = StandardScaler().fit(X)
X = stdScale.transform(X)
Y = df['Release']
G = df['DP_Group']

model = LGBMRegressor()



In [10]:
# Ward_linkage
corr = spearmanr(X).correlation # generate a correlation matrix is symmetric
corr = (corr + corr.T) / 2 # ensure the correlation matrix is symmetric
np.fill_diagonal(corr, 1)
distance_matrix = 1 - np.abs(corr) # convert the correlation matrix to a distance matrix 
dist_linkage = hierarchy.ward(squareform(distance_matrix)) # generate Ward's linkage values for hierarchical clustering

In [11]:
# evaluate_model iterations
MAE_list = [] # empty list to store MAE values
std_list = [] # empty list to store MAE values
test_test_list = [] # empty list to store MAE values
feature_name_list = [] # empty list to store features names
feature_number_list = [] # empty list to store number of features
linkage_distance_list = [] # empty list to store the Ward'slinkage distance
        
for n in range(0, 17, 1):
    cluster_ids = hierarchy.fcluster(dist_linkage, (n/17), criterion="distance") # select input features to be included in this model iteration based on Ward's linkage of n/10
    cluster_id_to_feature_ids = defaultdict(list) 
    
    for idx, cluster_id in enumerate(cluster_ids):
        cluster_id_to_feature_ids[cluster_id].append(idx)
            
    selected_features = [v[0] for v in cluster_id_to_feature_ids.values()]
    linkage_distance_list.append(n/17) # append linkage distance to empty list
    tested_features = []  # create empty list to save feature names
        
    for feature in selected_features: # for loop to append the utilized input feature names to the empty list
        tested_features.append(X_features.columns[feature])
            
    feature_number_list.append(len(tested_features)) # append the number of input features to empty list
    feature_name_list.append(tested_features) # append the list of feature names to an empty list of lists
            
    test_list = []

    for i in range(10): # for loop that splits and testd the model 10 times to generate mean and stdev values
        cv_outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=i+1) #hold back 20% of the groups for test set
            
        for train_index, test_index in cv_outer.split(X, Y, G): # split data using group-shuffle-split based on drug-polymer groups
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = Y[train_index], Y[test_index]
                    
            X_train_sel = X_train[:, selected_features] # select input features from training dataset based on Ward's Linkage value
            X_test_sel = X_test[:, selected_features] # select input features from test dataset based on Ward's Linkage value
                    
            clf_sel = model # assign selected model to clf_sel
            clf_sel.fit(X_train_sel, y_train) # fit the selected model with the training set
            y_pred = clf_sel.predict(X_test_sel) # predict test set based on selected input features
            test_list.append(round(mean_absolute_error(y_pred, y_test), 3)) # append average MAE value to empty list
            
    MAE_list.append(np.mean(test_list)) # append average MAE value to empty list
    std_list.append(np.std(test_list)) # append average MAE value to empty list
    test_test_list.append(test_list) # append average MAE value to empty list
                
    print('\n################################################################\n\nSTATUS REPORT:') 
    print('Iteration '+str(n+1)+' of '+str(17)+' completed') 
    print('Test_Score: %.3f' % (np.mean(test_list)))
    print("\n################################################################\n ")

C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000612 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 678
[LightGBM] [Info] Number of data points in the train set: 3237, number of used features: 17
[LightGBM] [Info] Start training from score 0.489447


C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007110 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 650
[LightGBM] [Info] Number of data points in the train set: 3077, number of used features: 17
[LightGBM] [Info] Start training from score 0.464761


C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 65
[LightGBM] [Info] Number of data points in the train set: 2984, number of used features: 4
[LightGBM] [Info] Start training from score 0.468066


C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 64
[LightGBM] [Info] Number of data points in the train set: 2260, number of used features: 4
[LightGBM] [Info] Start training from score 0.429204

################################################################

STATUS REPORT:
Iteration 17 of 17 completed
Test_Score: 0.301

################################################################
 


C:\Users\prajw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [12]:
# results dataframe 
list_of_tuples = list(zip(feature_number_list, feature_name_list, MAE_list, std_list, test_test_list, linkage_distance_list)) # create a list of tuples with results model refinement
results_df = pd.DataFrame(list_of_tuples, columns = ['# of Features', 'Feature names', 'MAE', 'std', 'test_values','linkage distance']) # create a dataframe with results model refinement
results_df

,# of Features,Feature names,MAE,std,test_values,linkage distance
0,17,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1267,0.017257,"[0.127, 0.101, 0.119, 0.123, 0.122, 0.129, 0.1...",0.000000
1,15,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1364,0.019397,"[0.14, 0.112, 0.14, 0.123, 0.124, 0.142, 0.107...",0.058824
2,13,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1562,0.025736,"[0.143, 0.14, 0.153, 0.144, 0.126, 0.151, 0.14...",0.117647
3,12,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1517,0.020494,"[0.131, 0.148, 0.142, 0.139, 0.127, 0.162, 0.1...",0.176471
4,12,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1517,0.020494,"[0.131, 0.148, 0.142, 0.139, 0.127, 0.162, 0.1...",0.235294
5,11,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1520,0.019718,"[0.133, 0.141, 0.14, 0.142, 0.125, 0.164, 0.14...",0.294118
6,10,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1519,0.019715,"[0.138, 0.13, 0.15, 0.14, 0.137, 0.156, 0.141,...",0.352941
7,10,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1519,0.019715,"[0.138, 0.13, 0.15, 0.14, 0.137, 0.156, 0.141,...",0.411765
8,10,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1519,0.019715,"[0.138, 0.13, 0.15, 0.14, 0.137, 0.156, 0.141,...",0.470588
9,9,"[LA/GA, Polymer_MW, CL Ratio, Drug_Tm, Drug_Pk...",0.1475,0.023385,"[0.129, 0.125, 0.157, 0.136, 0.123, 0.15, 0.13...",0.529412


In [13]:
results_df.to_excel("LGBM_15_model_refinement_results.xlsx")